## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [18]:
import seaborn as sns


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
65,1,3,male,NaN,1,1,15.2458,C,Third,man,True,NaN,Cherbourg,yes,False
208,1,3,female,16.0,0,0,7.7500,Q,Third,woman,False,NaN,Queenstown,yes,True
483,1,3,female,63.0,0,0,9.5875,S,Third,woman,False,NaN,Southampton,yes,True
219,0,2,male,30.0,0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
819,0,3,male,10.0,3,2,27.9000,S,Third,child,False,NaN,Southampton,no,False


In [19]:
import numpy as np  
import pandas as pd

**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [20]:
missing_values = titanic_data.isnull().sum()

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [7]:
rows = len(titanic_data)

miss_columns = titanic_data.isnull().sum()
mask_corr_colum = miss_columns <= rows / 2
titanic_clean = titanic_data.loc[:, mask_corr_colum]
n_cols = titanic_clean.shape[1]
titanic_clean = titanic_clean.dropna(thresh=int(np.ceil(n_cols / 2)))

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [8]:
who = titanic_clean["who"]
age = titanic_clean["age"]

median_man   = round(age[who == "man"].median())
median_woman = round(age[who == "woman"].median())
median_child = round(age[who == "child"].median())

fill_values = (
    (who == "man")   * median_man
    + (who == "woman") * median_woman
    + (who == "child") * median_child
)

titanic_clean["age"] = age.fillna(fill_values)


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [23]:
cols_num = titanic_clean.shape[1]
titanic_clean = titanic_clean.dropna(thresh=n_cols - 1)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [22]:
titanic_clean["embark_town"].value_counts().idxmax()

'Southampton'

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [12]:
round(titanic_clean["survived"].sum() / len(titanic_clean) * 100, 2)

np.float64(38.25)

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [24]:
survived_pass = titanic_clean.loc[titanic_clean["survived"] == 1]
survivors_by_town = survived_pass["embark_town"].value_counts()

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [25]:
class_col = titanic_clean["class"]
survived = titanic_clean["survived"]

result = {}
for cls in ["First", "Second", "Third"]:
    mask = class_col == cls
    result[cls] = round(survived[mask].mean() * 100, 2)

survived_pct_by_class = pd.Series(result)
print(survived_pct_by_class)

First     62.62
Second    47.28
Third     24.24
dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [ ]:
rich_mask = titanic_clean["fare"] >= 100
rich_survival = round(
    titanic_clean.loc[rich_mask, "survived"].mean() * 100,
    2,
)
print(rich_survival)

73.58


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [17]:
mask = (titanic_clean["who"] == "child") & titanic_clean["alone"]
children_alone = mask.sum()
print(children_alone)

6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 

Можно сказать, что много выживших среди тех, кто покупал дорогой билет (>70%), также много пассажиров начало свое путешествие из города Southampton.
Как оказалось на борту были дети, которые путешествовали в одиночку. Также я еще немного поигрался с данными вне ноутбука и узнал, что были пассажиры с ценой билета в 0